In [1]:
import requests
import numpy as np
import pandas as pd
import os,sys,glob
import matplotlib.pyplot as plt
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
from shutil import which
import geopandas as gpd

from pyproj import Proj, transform
import subprocess
from shapely.geometry import Point

import xyzservices
import rioxarray as rxr
#import rasterio
from shapely.geometry import box
import xarray as xr
from sliderule import sliderule, icesat2, earthdata
#import contextily as ctx
#import sliderule
#import xyzservices

In [2]:
def fetch_is2_atl08(workunit,raster_fn,gpkg_outfn):
    if 'usgs_workunit' in gf_is2_stac.columns:
        mask = gf_is2_stac['usgs_workunit'] == workunit
    elif 'neon_id' in gf_is2_stac.columns:
        mask = gf_is2_stac['neon_id'] == workunit
    else:
        mask = gf_is2_stac['ncalm_id'] == workunit
    is2_ids = [str(value)+'.h5' for value in gf_is2_stac[mask].is2_granule_id.values.tolist()]
    #is2_ids = [string.replace('ATL03', 'ATL08') for string in is2_ids]
    print(is2_ids)
    datetime = [str(np.sort(gf_is2_stac[mask].date.values)[0]), str(np.sort(gf_is2_stac[mask].date.values)[-1])]

    da = rxr.open_rasterio(raster_fn, masked=True).squeeze()
    da_bounds = da.rio.bounds()  # (minx, miny, maxx, maxy)
    da_box = box(*da_bounds)
    gf_search = gpd.GeoDataFrame(geometry=[da_box],
                                 columns=['geometry'],
                                 crs = da.rio.crs)
    gf_search = gf_search.to_crs(4326)
    
    poly_sliderule = sliderule.toregion(gf_search)['poly']
    #https://nbviewer.org/github/ICESat2-SlideRule/sliderule-python/blob/main/examples/phoreal.ipynb
    #gf_is2 = icesat2.atl06p(parms,
     #                       resources=is2_ids)

    gf_is2_atl08 = icesat2.atl08p({
    "poly": poly_sliderule,
    "t0": datetime[0],
    "t1": datetime[1]
    }, 
    resources = is2_ids,
    )
    
    gf_is2_atl08 = gf_is2_atl08.to_crs(gf_is2_atl08.estimate_utm_crs())
    gf_is2_atl08['easting'] = gf_is2_atl08.geometry.x
    gf_is2_atl08['northing'] = gf_is2_atl08.geometry.y
    dirname = os.path.dirname(gpkg_outfn)
    if not os.path.exists(dirname):
        os.makedirs(dirname)

    print(len(gf_is2_atl08))
    gf_is2_atl08.to_file(os.path.splitext(gpkg_outfn)[0]+"_atl08_sr.gpkg",driver='GPKG')
    
    ## Functions
def load_stv_product_local(href, overview_level=None):
    if overview_level is not None:
        da = xr.open_dataarray(
            href,
            engine="rasterio",
            mask_and_scale=True,
            backend_kwargs={"open_kwargs": {"overview_level": overview_level}},
        ).squeeze()
    else:
        da = xr.open_dataarray(
            href,
            engine="rasterio",
            mask_and_scale=True,
        ).squeeze()
    return da
    

In [3]:
%cd /panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/usgs

/panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/usgs


In [8]:
gf_is2_stac = gpd.read_file('/home/sbhusha1/notebooks/pcd/20250709_PCD_3dep_is2_tracks.geojson')
workunit_list = np.unique(gf_is2_stac.usgs_workunit.values).tolist()

In [9]:
for workunit in workunit_list:
    
    data_dir = glob.glob(f"{workunit}_*processing*/")[0]
    dtm_fn = glob.glob(os.path.join(data_dir,'*DTM*4*mos.tif'))[0]
    outdir = os.path.join(data_dir,"ICESat-2_SR")
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    outfn = os.path.join(outdir,f"{workunit}_IS2.gpkg")
    #print(dtm_fn,outfn)
    print(workunit)
    fetch_is2_atl08(workunit,dtm_fn,outfn)

AZ_PimaCo_2_2021
['ATL03_20211018062519_03941306_006_01.h5', 'ATL03_20211129162037_10421302_006_01.h5']
3983
CA_SanFrancisco_1_B23
['ATL03_20230117092727_04251806_006_02.h5', 'ATL03_20230418050714_04251906_006_02.h5', 'ATL03_20230522151905_09511902_006_02.h5', 'ATL03_20230718004632_04252006_006_02.h5']
4427
CA_YosemiteNP_2019
['ATL03_20191008182255_01810506_006_02.h5']
3285
CO_WestCentral_2019
['ATL03_20191008052407_01730502_006_02.h5']
10961
GA_Central_3_2019
['ATL03_20200203102127_05910606_006_01.h5', 'ATL03_20200316201642_12390602_006_01.h5']
21665
NE_Northeast_Phase2_2_2020
['ATL03_20201119213030_08660906_006_01.h5', 'ATL03_20201220075046_13310902_006_01.h5']
24305
TX_DesertMountains_B1_2018
['ATL03_20190924180456_13540406_006_02.h5']
12756
WI_Brown_2_2020
['ATL03_20200503062653_05760706_006_01.h5']
3008
WY_FEMA_East_B9_2019
['ATL03_20190721091907_03560402_006_02.h5', 'ATL03_20190723212639_03940406_006_02.h5', 'ATL03_20190819075518_07980402_006_02.h5', 'ATL03_20190821200249_0836040

### Check co-reg potential

In [12]:
sample_atl08 = gpd.read_file('WI_Brown_2_2020_processing/ICESat-2_SR/WI_Brown_2_2020_IS2_atl08_sr.gpkg')

In [13]:
sample_atl08.head(2)

,time,h_min_canopy,ph_count,cycle,x_atc,h_mean_canopy,h_canopy,canopy_h_metrics,canopy_openness,gt,...,spot,rgt,h_te_median,gnd_ph_count,snowcover,solar_elevation,landcover,easting,northing,geometry
0,2020-05-03 06:30:46.561,-7.120412e+14,28,7,15097862.0,9.183269e-41,-6.388903e+14,"(0.0, 9.183269356106256e-41, -34012.0, -930359...",9.183269e-41,10,...,6,576,2.770367e-42,3626100296,255,-28.747663,255,417880.584196,4.947153e+06,POINT (417880.584 4947153.067)
1,2020-05-03 06:30:46.566,-7.120412e+14,26,7,15097892.0,9.183269e-41,-6.388903e+14,"(0.0, 9.183269356106256e-41, -34012.0, -930359...",9.183269e-41,10,...,6,576,2.767564e-42,3626100296,255,-28.747852,255,417877.326715,4.947123e+06,POINT (417877.327 4947123.151)


In [14]:
sample_atl08.keys()

Index(['time', 'h_min_canopy', 'ph_count', 'cycle', 'x_atc', 'h_mean_canopy',
       'h_canopy', 'canopy_h_metrics', 'canopy_openness', 'gt', 'veg_ph_count',
       'segment_id', 'h_max_canopy', 'spot', 'rgt', 'h_te_median',
       'gnd_ph_count', 'snowcover', 'solar_elevation', 'landcover', 'easting',
       'northing', 'geometry'],
      dtype='object')

In [9]:
def adhoc_atl08_alignment(log_fn,gf_is2,elevation_col,aligned_gpkg):
    # Parse alignment results
    #log_files = sorted(glob.glob(f"{align_prefix}*log*pc_align*.txt"))
    #if not log_files:
     #   print("No log file found")
     #   return gf_is2, _empty_record("no_log_found")
    
    params = parse_pc_align_log(log_fn)
    dx, dy, dz = params["translation_enu"]
    
    print(
        f"Alignment result: dx={dx:.2f} m, dy={dy:.2f} m, "
        f"dz={dz:.2f} m, total={params['total_displacement']:.2f} m"
    )
    
    # Apply shift
    aligned_gdf = apply_shift(gf_is2, dx, dy, dz, elevation_cols=elevation_col)
    
    #aligned_gpkg = f"{args.outprefix}-IS2_ground_aligned.gpkg"
    aligned_gdf.to_file(aligned_gpkg, driver="GPKG")
    print(f"Aligned footprints saved: {aligned_gpkg}")

In [20]:
for workunit in workunit_list:
    
    data_dir = glob.glob(f"{workunit}_*processing*/")[0]
    dtm_fn = glob.glob(os.path.join(data_dir,'*DTM*4*mos.tif'))[0]
    outdir = os.path.join(data_dir,"ICESat-2_SR")
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    og_fn = os.path.join(outdir,f"{workunit}_IS2.gpkg")
    atl08_fn = os.path.splitext(og_fn)[0]+"_atl08_sr.gpkg"
    atl08_aligned_fn = os.path.splitext(atl08_fn)[0]+"_aligned.gpkg"
    log_fn = sorted(glob.glob(os.path.join(outdir,'*align_is2/*log-pc_align*.txt')))[-1]
    #print(dtm_fn,outfn)
    print(workunit)
    adhoc_atl08_alignment(log_fn,gpd.read_file(atl08_fn),'h_te_median',atl08_aligned_fn) 

AZ_PimaCo_2_2021
Alignment result: dx=-1.67 m, dy=1.65 m, dz=-0.08 m, total=2.35 m
Aligned footprints saved: AZ_PimaCo_2_2021_processing/ICESat-2_SR/AZ_PimaCo_2_2021_IS2_atl08_sr_aligned.gpkg
CA_SanFrancisco_1_B23
Alignment result: dx=-2.82 m, dy=0.90 m, dz=-0.01 m, total=2.96 m
Aligned footprints saved: CA_SanFrancisco_1_B23_processing/ICESat-2_SR/CA_SanFrancisco_1_B23_IS2_atl08_sr_aligned.gpkg
CA_YosemiteNP_2019
Alignment result: dx=-2.30 m, dy=1.39 m, dz=0.42 m, total=2.72 m
Aligned footprints saved: CA_YosemiteNP_2019_processing/ICESat-2_SR/CA_YosemiteNP_2019_IS2_atl08_sr_aligned.gpkg
CO_WestCentral_2019
Alignment result: dx=-1.30 m, dy=0.94 m, dz=0.12 m, total=1.60 m
Aligned footprints saved: CO_WestCentral_2019_processing/ICESat-2_SR/CO_WestCentral_2019_IS2_atl08_sr_aligned.gpkg
GA_Central_3_2019
Alignment result: dx=0.82 m, dy=0.70 m, dz=0.03 m, total=1.08 m
Aligned footprints saved: GA_Central_3_2019_processing/ICESat-2_SR/GA_Central_3_2019_IS2_atl08_sr_aligned.gpkg
NE_Northeas

## NEON

In [21]:
%cd /panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/neon

/panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/neon


In [22]:
gf_is2_stac = gpd.read_file('/home/sbhusha1/notebooks/pcd/20250718_PCD_neon_is2_tracks.geojson')
workunit_list = np.unique(gf_is2_stac.neon_id.values).tolist()

In [23]:
for workunit in workunit_list:
    
    data_dir = glob.glob(f"{workunit}_*processing*/")[0]
    dtm_fn = glob.glob(os.path.join(data_dir,'*DTM*4*mos.tif'))[0]
    outdir = os.path.join(data_dir,"ICESat-2_SR")
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    outfn = os.path.join(outdir,f"{workunit}_IS2.gpkg")
    #print(dtm_fn,outfn)
    print(workunit)
    fetch_is2_atl08(workunit,dtm_fn,outfn)

BART
['ATL03_20190903043904_10250402_006_02.h5', 'ATL03_20190807181029_06210406_006_02.h5']
1891
REDB
['ATL03_20210628115237_00741206_006_01.h5']
1305
WREF
['ATL03_20190529005739_09280306_006_02.h5', 'ATL03_20190522125829_08290302_006_02.h5', 'ATL03_20190821083817_08290402_006_02.h5']
1444


In [24]:
for workunit in workunit_list:
    
    data_dir = glob.glob(f"{workunit}_*processing*/")[0]
    dtm_fn = glob.glob(os.path.join(data_dir,'*DTM*4*mos.tif'))[0]
    outdir = os.path.join(data_dir,"ICESat-2_SR")
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    og_fn = os.path.join(outdir,f"{workunit}_IS2.gpkg")
    atl08_fn = os.path.splitext(og_fn)[0]+"_atl08_sr.gpkg"
    atl08_aligned_fn = os.path.splitext(atl08_fn)[0]+"_aligned.gpkg"
    log_fn = sorted(glob.glob(os.path.join(outdir,'*align_is2/*log-pc_align*.txt')))[-1]
    #print(dtm_fn,outfn)
    print(workunit)
    adhoc_atl08_alignment(log_fn,gpd.read_file(atl08_fn),'h_te_median',atl08_aligned_fn) 

BART
Alignment result: dx=0.59 m, dy=-0.85 m, dz=0.82 m, total=1.32 m
Aligned footprints saved: BART_first_idw_processing/ICESat-2_SR/BART_IS2_atl08_sr_aligned.gpkg
REDB
Alignment result: dx=-2.87 m, dy=-0.22 m, dz=1.34 m, total=3.17 m
Aligned footprints saved: REDB_first_idw_processing/ICESat-2_SR/REDB_IS2_atl08_sr_aligned.gpkg
WREF
Alignment result: dx=-2.96 m, dy=1.11 m, dz=0.28 m, total=3.17 m
Aligned footprints saved: WREF_first_idw_processing/ICESat-2_SR/WREF_IS2_atl08_sr_aligned.gpkg


## NCALM

In [4]:
%cd /panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/ncalm/

/panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/ncalm


In [5]:
gf_is2_stac = gpd.read_file('/home/sbhusha1/notebooks/pcd/20250722_PCD_ncalm_is2_tracks.geojson')
workunit_list = np.unique(gf_is2_stac.ncalm_id.values).tolist()

In [10]:


for workunit in workunit_list:
    
    data_dir = glob.glob(f"SA_Fault_processing/")[0]
    dtm_fn = glob.glob(os.path.join(data_dir,'*dsm*1m.tif'))[0]
    outdir = os.path.join(data_dir,"ICESat-2_SR")
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    outfn = os.path.join(outdir,f"SA_Fault_IS2.gpkg")
    #print(dtm_fn,outfn)
    print(workunit)
    fetch_is2_atl08(workunit,dtm_fn,outfn)

for workunit in workunit_list:
    
    data_dir = glob.glob(f"SA_Fault_processing/")[0]
    dtm_fn = glob.glob(os.path.join(data_dir,'*dsm*1m.tif'))[0]
    outdir = os.path.join(data_dir,"ICESat-2_SR")
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    og_fn = os.path.join(outdir,f"SA_Fault_IS2.gpkg")
    atl08_fn = os.path.splitext(og_fn)[0]+"_atl08_sr.gpkg"
    atl08_aligned_fn = os.path.splitext(atl08_fn)[0]+"_aligned.gpkg"
    log_fn = sorted(glob.glob(os.path.join(outdir,'*align_is2/*log-pc_align*.txt')))[-1]
    #print(dtm_fn,outfn)
    print(workunit)
    adhoc_atl08_alignment(log_fn,gpd.read_file(atl08_fn),'h_te_median',atl08_aligned_fn) 

OTLAS.092021.32611.1
['ATL03_20200126004754_04630602_006_01.h5', 'ATL03_20200116132021_03180606_006_01.h5']


Alert <-7>: Failure on resource ATL03_20200116132021_03180606_006_01.h5 track 3.0: H5Coro::Future read failure on /gt3l/geolocation/segment_ph_cnt
Alert <-7>: Failure on resource ATL03_20200116132021_03180606_006_01.h5 track 2.1: H5Coro::Future read failure on /gt2r/geolocation/segment_id


2272
OTLAS.092021.32611.1
Alignment result: dx=-0.73 m, dy=2.21 m, dz=0.03 m, total=2.33 m
Aligned footprints saved: SA_Fault_processing/ICESat-2_SR/SA_Fault_IS2_atl08_sr_aligned.gpkg


## Alignment Functions

In [7]:
def parse_pc_align_log(log_path):
    """
    Parse pc_align log file for translation, rotation, scale, and total shift.

    Parameters
    ----------
    log_path : str
        Path to pc_align log file.

    Returns
    -------
    dict with keys:
        total_displacement : float (meters)
        translation_ned : tuple (north, east, down) in meters
        translation_enu : tuple (east, north, up) in meters
        euler_angles_ned : tuple (north, east, down) in degrees
        scale : float
    """
    with open(log_path, "r") as f:
        content = f.readlines()

    # Total displacement
    substr = (
        "Maximum displacement of points between the source cloud with any "
        "initial transform applied to it and the source cloud after alignment "
        "to the reference"
    )
    match = [line for line in content if substr in line]
    total_displacement = float(match[0].split(":", 15)[-1].split("m")[0])

    # Translation vector (North-East-Down)
    substr = "Translation vector (North-East-Down, meters):"
    line = [l for l in content if substr in l][0]
    vec = line.split("Vector3")[1]
    north = float(vec.split("(")[1].split(",")[0])
    east = float(vec.split(",")[1])
    down = float(vec.split(",")[2].split(")")[0])

    # Euler angles (North-East-Down)
    substr = " Euler angles (North-East-Down, degrees)"
    line = [l for l in content if substr in l][0]
    vec = line.split("Vector3")[1]
    north_angle = float(vec.split("(")[1].split(",")[0])
    east_angle = float(vec.split(",")[1])
    down_angle = float(vec.split(",")[2].split(")")[0])

    # Scale
    substr = "Transform scale"
    line = [l for l in content if substr in l][0]
    scale = float(line.split("= ")[1].split("\n")[0]) + 1

    return {
        "total_displacement": total_displacement,
        "translation_ned": (north, east, down),
        "translation_enu": (east, north, -1 * down),
        "euler_angles_ned": (north_angle, east_angle, down_angle),
        "scale": scale,
    }
def apply_shift(gdf, dx, dy, dz, elevation_cols=None):
    """
    Apply a rigid translation to footprint positions and elevations.

    Parameters
    ----------
    gdf : GeoDataFrame
        Input footprints with geometry and elevation column(s).
    dx, dy, dz : float
        Easting, northing, and vertical shifts in meters.
    elevation_cols : str or list of str, optional
        Name(s) of elevation column(s) to shift vertically.
        If None, no elevation columns are shifted.

    Returns
    -------
    GeoDataFrame
        Shifted footprints with updated geometry and elevation(s).
    """
    if elevation_cols is None:
        elevation_cols = []
    elif isinstance(elevation_cols, str):
        elevation_cols = [elevation_cols]

    crs = gdf.estimate_utm_crs()
    out = gdf.copy()
    for col in elevation_cols:
        if col in out.columns:
            out[col] = gdf[col] + dz
        else:
            print(f"  WARNING: elevation column '{col}' not found, skipping vertical shift for it")
    out["easting"] = gdf.geometry.x + dx
    out["northing"] = gdf.geometry.y + dy
    geometry = [Point(x, y) for x, y in zip(out["easting"], out["northing"])]
    out = gpd.GeoDataFrame(out.drop(columns=["geometry"]), geometry=geometry, crs=crs)
    return out